# TorchVision para Pre-Processing

¡Bienvenido al primer lab de este módulo! Antes de que un modelo de deep learning pueda aprender a "ver", los datos visuales que le proporcionas deben ser preparados cuidadosamente. Las imágenes crudas vienen en varios tamaños y formatos, pero las redes neuronales requieren una entrada estandarizada, específicamente, un **tensor**.

Aquí es donde entra **TorchVision**. Como el toolkit estándar de PyTorch para computer vision, proporciona herramientas potentes y eficientes diseñadas para manejar los componentes comunes, y a menudo tediosos, de una carga de trabajo de visión. En lugar de reinventar la rueda, puedes usar los pipelines de datos y funciones de TorchVision, probados en batalla, para enfocarte en construir modelos innovadores.

En este lab, podrás:

* Practicar la conversión de imágenes entre el formato común de imagen **Pillow (PIL)** y los **PyTorch Tensors**.
* Explorar utilidades útiles de **TorchVision** como `make_grid` y `save_image` para simplificar el debugging y visualizar batches de imágenes.
* Aplicar transformaciones individuales para hacer resize, crop y augment de imágenes, y ver sus efectos.
* Definir e implementar una **transformación personalizada** desde cero para añadir un efecto único al dataset.
* Encadenar transformaciones usando `transforms.Compose` para construir **pipelines de preprocessing y data augmentation** potentes y reutilizables.

Al final de este lab, tendrás una comprensión sólida de cómo construir un workflow completo de preparación de imágenes desde cero.

## Imports

In [ ]:
import os

from IPython.display import Image as DisplayImage
import numpy as np
from PIL import Image
import torch
import torch.utils.data as data
from torchvision import datasets
from torchvision.io import decode_image
import torchvision.transforms as transforms
import torchvision.utils as vutils
from tqdm.auto import tqdm

import helper_utils

In [ ]:
# Check if the OxfordIIITPet data folder exists
ox3_pet_data_path = './oxford3pet_data'
if os.path.exists(ox3_pet_data_path) and os.path.isdir(ox3_pet_data_path):
    ox3_pet_download = False  # Data folder exists, will be loaded from
else:
    ox3_pet_download = True  # Data folder doesn't exist, will be downloaded

## Image Conversion (PIL and Tensor)

Tu primer paso práctico en cualquier workflow de computer vision es cerrar la brecha entre cómo los humanos ven las imágenes y cómo las máquinas las procesan. Un archivo de imagen es una cuadrícula de píxeles, a menudo manejada por una librería como Pillow (PIL). Una red neuronal, sin embargo, requiere un formato numérico sobre el cual pueda realizar cálculos: un **Tensor**.

Ganar fluidez en la conversión entre estos dos formatos es una habilidad fundamental. Es el mecanismo que te permite cargar, procesar e inspeccionar tus datos en cada etapa de un proyecto.

Para comenzar, realizarás una conversión de "ida y vuelta" (roundtrip). Al tomar una imagen PIL, cambiarla a un PyTorch Tensor y luego convertirla de nuevo, confirmarás que esta operación principal es fluida y preserva tus datos perfectamente. Este es un sanity check vital antes de construir pipelines más complejos.

* Carga una imagen con Pillow desde un archivo usando la librería Pillow (PIL).
    * Ten en cuenta que Pillow reporta las dimensiones en formato `(Width, Height)`.

In [ ]:
# Load an image
image = Image.open('./images/mangoes.jpg')

# Dimensions of the original PIL image
print("Original PIL Image Dimensions:", image.size)
print(f"The maximum pixel value is: {image.getextrema()[0][1]}, and the minimum is: {image.getextrema()[0][0]}")

<br>

* `transforms.ToTensor()`: Convierte un objeto de imagen PIL en un PyTorch Tensor.
    * **Dimension Change**: Esta transform reordena los datos de la imagen del formato `(Width, Height)` de Pillow al formato `(Channels, Height, Width)` de PyTorch. También escala los valores de los píxeles de la imagen del rango `[0, 255]` a un rango de punto flotante `[0.0, 1.0]`.

In [ ]:
# Convirtiendo la imagen PIL a un tensor de PyTorch
img_tensor = transforms.ToTensor()(image)

# Dimensiones (forma) del tensor en formato
# [C, H, W] format
print(f"Dimensiones después de convertir a un tensor: {img_tensor.shape}")
print(f"El valor máximo de píxel es: {img_tensor.max()}, y el mínimo es: {img_tensor.min()}")

<br>

* `transforms.ToPILImage()`: Convierte un PyTorch Tensor de nuevo a un objeto de imagen PIL.
    * **Dimension Change**: Realiza lo inverso, convirtiendo un tensor de `(Channels, Height, Width)` de nuevo en una imagen PIL que reporta su tamaño como `(Width, Height)`.

In [ ]:
# Convertimos el tensor de vuelta a una imagen PIL
img_pil = transforms.ToPILImage()(img_tensor)

# Dimensiones de la imagen PIL convertida de vuelta
print("Dimensiones después de convertir de nuevo a PIL:", img_pil.size)

<br>

* Muestra la imagen original y la convertida una al lado de la otra para confirmar visualmente que el proceso de "ida y vuelta" (roundtrip) preservó el contenido de la imagen perfectamente.

In [ ]:
# Visualizar las imágenes original y convertida
helper_utils.show_images([image, img_pil], titles=("Original Image", "After PIL→Tensor→PIL conversion"))

## TorchVision Utilities for Image Handling

TorchVision te equipa con un toolkit potente para la logística práctica de un proyecto de computer vision. Estas utilidades están diseñadas para gestionar todo el ciclo de vida de tus datos de imagen, desde la carga inicial hasta la salida final. Dominarlas te permite cargar datos eficientemente, hacer debug de tu pipeline visualizando lo que tu modelo está viendo y guardar tus resultados de manera profesional.



Ahora explorarás tres funciones indispensables que abordan estas necesidades principales:

* `decode_image`: Convierte instantáneamente archivos de imagen comprimidos como JPEGs o PNGs directamente en tensors.

* `make_grid`: Organiza un batch de imágenes en una cuadrícula limpia y única, lo cual es perfecto para la inspección y el análisis de un vistazo.

* `save_image`: Guarda tus imágenes basadas en tensors de nuevo en un formato de archivo estándar, facilitando el compartir resultados para reportes o presentaciones.

### Decoding Images into Tensors con `decode_image`

La función `decode_image` está diseñada para un propósito principal: convertir eficientemente un archivo de imagen directamente en un **numerical PyTorch tensor** para computación. Lee una imagen y devuelve inmediatamente un tensor en formato `[Channels, Height, Width]`, listo para el procesamiento posterior. A diferencia de `Image.open`, que devuelve un objeto visual PIL para visualización o edición inmediata, el tensor de `decode_image` es un objeto puramente numérico. Esto lo convierte en la elección ideal cuando tu workflow comienza con computación, no con visualización.

* <code>[decode_image()](https://docs.pytorch.org/vision/stable/generated/torchvision.io.decode_image.html)</code>: Carga una imagen (e.g., JPEG, PNG) y la convierte en un PyTorch Tensor de tipo `torch.uint8`.
    * **Dimension Ordering**: El tensor de salida sigue la convención estándar de PyTorch de **channel-first** (`[C, H, W]`), lo cual es importante para la compatibilidad con el modelo. 'C' es el número de canales (e.g., 3 para una imagen RGB), 'H' es la altura (height) y 'W' es el ancho (width).

In [ ]:
# Define the path to the image file.
image_path = './images/apples.jpg'

# Load the image
image = decode_image(image_path)

print(f"Image tensor dimensions: {image.shape}")
print(f"Image tensor dtype: {image.dtype}")
print(f"The maximum pixel value is: {image.max()}, and the minimum is: {image.min()}\n")

In [ ]:
# Use the DisplayImage to render the image
DisplayImage(image_path, width=500, height=500)

### Creating Image Grids con `make_grid`

En deep learning, casi siempre procesas datos en **batches**, no imágenes individuales. Pero, ¿cómo obtienes una vista rápida y holística de un batch completo a la vez? Mostrarlas individualmente es ineficiente.

La función `make_grid` es la solución profesional a este desafío común. Es una utilidad esencial que toma un batch de tensors de imagen y los organiza en una única cuadrícula limpia para una inspección sencilla. Este es un paso vital para el visual debugging, permitiéndote verificar instantáneamente los resultados de tu data augmentation o ver exactamente qué es lo que un `DataLoader` le está entregando a tu modelo.

* Carga un batch de imágenes desde una carpeta local (`"./images/"`).
    * Las imágenes se cargan como un único tensor que contiene todos los datos de las imágenes, que es lo que espera la función `make_grid`.

In [ ]:
# Crear un batch de imágenes (./images/ contiene solo 6 imágenes). Las imágenes se cargan como 300x300 píxeles
images_tensor = helper_utils.load_images("./images/")

# El tamaño es 6 imágenes x 3 canales de color x 300 píxeles de altura x 300 píxeles de ancho
print(f"Dimensiones del tensor de imágenes: {images_tensor.shape}")

* Organiza las imágenes cargadas en una cuadrícula utilizando la función <code>[make_grid()](https://docs.pytorch.org/vision/main/generated/torchvision.utils.make_grid.html)</code>.
    * `tensor:` Este debe ser un batch de imágenes, como un **tensor 4D** (`(B x C x H x W)`), para ser colocado en la cuadrícula.
    * `nrow=3`: Organiza las imágenes con **3 imágenes en cada fila**.
    * `padding=5`: Añade **5 píxeles** de espacio entre las imágenes.
    * `normalize=True`: Cambia los valores de los píxeles de la imagen a un rango estándar (0 a 1) para una visualización consistente.

Siéntete libre de modificar los parámetros y observa cómo esto afecta a la imagen.

In [ ]:
# Crear una cuadrícula a partir de las imágenes cargadas (2 filas de 3 para 6 imágenes)
grid = vutils.make_grid(tensor=images_tensor, nrow=3, padding=5, normalize=True)

# la forma (shape) proviene de 
# num_images/nrow*pixel_height+(num_images/nrow+1)*padding = 2*300+3*5 = 615
# nrow*pixel_width+(nrow+1)*padding = 3*300+4*5 = 920
print(f"Dimensiones del tensor de la imagen: {grid.shape}")
print(f"El valor máximo de píxel es: {grid.max()}, y el mínimo es: {grid.min()}\n")

# Mostrar la cuadrícula de imágenes usando una función auxiliar
helper_utils.display_grid(grid)

### Saving Tensors as Images con `save_image`

Tu trabajo dentro de un entorno de PyTorch, ya sea una cuadrícula de datos que hayas creado para debugging o una imagen generada por un modelo, existe como un tensor. Para que este trabajo sea tangible y se pueda compartir en reportes, presentaciones o para uso futuro, necesitas exportarlo de nuevo a un archivo de imagen estándar.

La función `save_image` es la solución directa para este paso final. Toma tu tensor y lo guarda eficientemente como un archivo de imagen de alta calidad, como un PNG o JPG, completando el "ida y vuelta" (roundtrip) de archivo a tensor y viceversa.

* <code>[save_image()](https://docs.pytorch.org/vision/main/generated/torchvision.utils.save_image.html)</code>: Guarda el tensor de la imagen en un archivo.
    * `tensor`: El tensor de la(s) imagen(es) que se va(n) a guardar.
    * `fp`: La ruta del archivo (file path), con el nombre y formato para la imagen de salida.

Puedes observar el archivo de imagen en la barra lateral izquierda.

In [ ]:
# Define the path to save the image file.
image_path = "./fruits_grid.png"

# Save the grid as a PNG image
vutils.save_image(tensor=grid, fp=image_path)

* Una vez que el tensor ha sido guardado en un archivo usando `save_image`, puede ser tratado como cualquier imagen regular y visualizado.

In [ ]:
# Use the DisplayImage to render the image
DisplayImage(image_path)

## Image Transformations and Data Augmentation

Las transformaciones de imagen son una parte fundamental de la preparación de datos para una red neuronal. Las utilizas no solo para estandarizar el tamaño y el formato de las imágenes, sino también para realizar **data augmentation**. El augmentation aumenta artificialmente la diversidad de tus datos de entrenamiento al crear versiones modificadas de las imágenes existentes. Esto ayuda al modelo a volverse más robusto y a generalizar mejor ante datos nuevos y no vistos.

La clave que debes recordar es que el **orden en el que aplicas estas transformaciones importa**. Una práctica común y efectiva es aplicar primero transformaciones geométricas (como resizing y cropping), luego color y otros augmentations, y finalmente, convertir la imagen a un tensor y normalizarla. Esto asegura que los augmentations se apliquen de manera consistente y que los datos finales estén en el formato correcto para el modelo.

### A Closer Look at Individual Transformations

Antes de combinar las transformaciones en un pipeline potente, examina algunas de las más comunes individualmente. Comprender el efecto específico de cada transform es importante para construir una estrategia de data augmentation efectiva.

A medida que explores estas diferentes técnicas de transformación, te animamos a experimentar con varias configuraciones para observar sus distintos resultados y cómo podrían impactar en el rendimiento de tu modelo.

In [ ]:
original_image = Image.open('./images/strawberries.jpg')

#### Resize

La transform [Resize](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Resize.html) es un paso de preprocessing común para asegurar que todas las imágenes en un batch tengan las mismas dimensiones.

* Reescala una imagen PIL de entrada a un `size` (tamaño) deseado.
    * `size`: El tamaño de salida objetivo.

In [ ]:
# Definir la transformación de resize (cuadrado de 50x50)
resize_transform = transforms.Resize(size=50)

# Aplicar la transformación
resized_image = resize_transform(original_image)

In [ ]:
# (Width, Height)
print(f"Original Dimensions: {original_image.size}")
print(f"Resized Dimensions:  {resized_image.size}\n")

helper_utils.show_images(
    images=[original_image, resized_image], 
    titles=("Original", "Resized to (50, 50)")
)

#### CenterCrop

La transform [CenterCrop](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.CenterCrop.html) se utiliza para enfocarse en la parte central de una imagen, eliminando el fondo que pueda distraer en los bordes.

* Extrae un parche cuadrado del centro de la imagen.
    * `size`: Un entero que define la altura (height) y el ancho (width) del recorte deseado.

In [ ]:
# Definir la transformación de center crop (256x256)
center_crop_transform = transforms.CenterCrop(size=256)

# Aplicar la transformación
cropped_image = center_crop_transform(original_image)

In [ ]:
# (Width, Height)
print(f"Original Dimensions: {original_image.size}")
print(f"Cropped Dimensions:  {cropped_image.size}\n")

helper_utils.show_images(
    images=[original_image, cropped_image],
    titles=("Original", "Center Crop (256, 256)")
)

#### RandomResizedCrop

La transform [RandomResizedCrop](https://pytorch.org/vision/stable/generated/torchvision.transforms.RandomResizedCrop.html) es una técnica de data augmentation que recorta aleatoriamente una porción de la imagen y luego le cambia el tamaño a un valor dado. Esto ayuda al modelo a volverse más robusto ante las variaciones en la escala y posición de los objetos dentro de la imagen. Debido a que implica una selección aleatoria, cada vez que apliques esta transform, es probable que obtengas una versión recortada y redimensionada ligeramente diferente de la imagen.

* Recorta una porción aleatoria de una imagen y la redimensiona a un tamaño especificado.
    * `size`: El tamaño de salida objetivo.

In [ ]:
# Define the RandomResizedCrop transformation (224x224)
random_resized_crop_transform = transforms.RandomResizedCrop(size=224)

# Apply the transformation
cropped_resized_image_1 = random_resized_crop_transform(original_image)
cropped_resized_image_2 = random_resized_crop_transform(original_image)
cropped_resized_image_3 = random_resized_crop_transform(original_image)

In [ ]:
# (Width, Height)
print(f"Original Dimensions: {original_image.size}")
print(f"RandomResizedCrop 1 Dimensions:  {cropped_resized_image_1.size}")
print(f"RandomResizedCrop 2 Dimensions:  {cropped_resized_image_2.size}")
print(f"RandomResizedCrop 3 Dimensions:  {cropped_resized_image_3.size}\n")

helper_utils.show_images(
    images=[original_image, cropped_resized_image_1],
    titles=("Original (2048, 2048)", "RandomResizedCrop 1 (224, 224)")
)
helper_utils.show_images(
    images=[cropped_resized_image_2, cropped_resized_image_3],
    titles=("RandomResizedCrop 2 (224, 224)", "RandomResizedCrop 3 (224, 224)")
)


#### RandomHorizontalFlip

[RandomHorizontalFlip](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.RandomHorizontalFlip.html) es una técnica de data augmentation que voltea la imagen horizontalmente de forma aleatoria. Le enseña al modelo que la identidad de un objeto no cambia si está reflejada como en un espejo.


* Voltea la imagen horizontalmente con una probabilidad dada.
    * `p`: La probabilidad de que se aplique el flip. El valor por defecto es 0.5.

In [ ]:
# Definir la transformación de horizontal flip
# Establecer p=1.0 para garantizar que el flip ocurra en esta demostración
flip_transform = transforms.RandomHorizontalFlip(p=1.0)

# Aplicar la transformación
flipped_image = flip_transform(original_image)

In [ ]:
helper_utils.show_images(
    images=[original_image, flipped_image],
    titles=("Original", "RandomHorizontalFlip (p=1.0)")
)

#### ColorJitter

[ColorJitter](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.ColorJitter.html), una técnica de data augmentation, hace que el modelo sea más robusto a las variaciones de iluminación y color al alterar aleatoriamente las propiedades de color de la imagen.

* Cambia aleatoriamente el brillo (brightness), contraste (contrast) y saturación (saturation) de una imagen.
    * Los parámetros `brightness`, `contrast` y `saturation` controlan el rango de los ajustes aleatorios.

In [ ]:
# Definir la transformación ColorJitter
# Los valores determinan el rango aleatorio para cada propiedad.
jitter_transform = transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)

# Aplicar la transformación
jittered_image = jitter_transform(original_image)

In [ ]:
helper_utils.show_images(
    images=[original_image, jittered_image],
    titles=("Original", "ColorJitter")
)

#### Custom Transformations

En algunos casos, es posible que necesites una operación específica que no esté integrada, como simular un tipo particular de ruido de cámara. En estas situaciones, puedes crear tus propias transformaciones personalizadas.

* Una transformación personalizada se crea definiendo una clase de Python con un método `__call__`.
    * Este método recibe una imagen PIL como entrada y devuelve la imagen modificada, permitiendo que se integre perfectamente en un pipeline de `transforms.Compose`.
* Define una transformación para aplicar ruido de "sal y pimienta" (salt and pepper noise), que añade píxeles aleatorios blancos (255) y negros (0) a una imagen. Esta transformación en particular se considera una técnica de data augmentation.

In [ ]:
class SaltAndPepperNoise:
    """
    Una transformación personalizada para añadir ruido de sal y pimienta a una imagen PIL.

    Args:
        salt_vs_pepper (float): La proporción de ruido de sal frente al de pimienta.
                                (e.g., 0.5 es una cantidad igual de cada uno).
        amount (float): La proporción total de píxeles que se verán afectados por el ruido.
    """
    def __init__(self, salt_vs_pepper=0.5, amount=0.04):
        self.s_vs_p = salt_vs_pepper
        self.amount = amount

    def __call__(self, image):
        # Hacer una copia de la imagen
        output = np.copy(np.array(image))

        # Añadir ruido de sal (Salt Noise)
        num_salt = np.ceil(self.amount * image.size[0] * image.size[1] * self.s_vs_p)
        # Generar coordenadas aleatorias para el ruido de sal
        coords = [np.random.randint(0, i - 1, int(num_salt)) for i in image.size]
        # Establecer los píxeles a blanco
        output[coords[1], coords[0]] = 255  

        # Añadir ruido de pimienta (Pepper Noise)
        num_pepper = np.ceil(self.amount * image.size[0] * image.size[1] * (1.0 - self.s_vs_p))
        # Generar coordenadas aleatorias para el ruido de pimienta
        coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in image.size]
        # Establecer los píxeles a negro
        output[coords[1], coords[0]] = 0

        # Convertir el array de NumPy de nuevo a una imagen PIL
        return Image.fromarray(output)

    def __repr__(self):
        return self.__class__.__name__ + f'(salt_vs_pepper={self.s_vs_p}, amount={self.amount})'

In [ ]:
# Instantiate your custom transformation
sp_transform = SaltAndPepperNoise(salt_vs_pepper=0.5, amount=0.5)

# Apply the transformation
sp_image = sp_transform(original_image)

In [ ]:
helper_utils.show_images(
    images=[original_image, sp_image],
    titles=("Original", "With Salt & Pepper Noise")
)

#### Normalize

[Normalize](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html?highlight=normalize) es un paso de preprocesamiento que estandariza los valores de los píxeles de una imagen. Resta la media y divide por la desviación estándar para cada canal. Esto ayuda a que el modelo converja más rápido durante el entrenamiento.

* Opera sobre un tensor de imagen, restando la media y dividiendo por la desviación estándar de cada canal.
    * `mean`: Una secuencia de valores medios para cada canal.
    * `std`: Una secuencia de valores de desviación estándar para cada canal.

**Nota**: `transforms.ToTensor()` siempre debe aplicarse antes de esta transformación, ya que esta opera sobre tensors, no sobre imágenes PIL.

In [ ]:
# Convertir a tensor (escala a [0, 1])
tensor_image = transforms.ToTensor()(original_image)

# Definir la transformación de normalización usando las estadísticas de ImageNet
normalize_transform = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# Aplicar la transformación
normalized_tensor = normalize_transform(tensor_image)

In [ ]:
# Visualize the distribution before and after normalization
helper_utils.plot_histogram(tensor_image, normalized_tensor, "Comparison of Pixel Distribution Before and After Normalization")

##### Calculating Dataset Mean and Standard Deviation

Al aplicar la transformación de normalización, utilizaste la media y la desviación estándar bien conocidas del dataset ImageNet:

> `mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]`

Usar la media y la desviación estándar de un dataset grande como ImageNet es una práctica común y altamente efectiva, especialmente cuando se realiza el fine-tuning de un modelo pre-entrenado. Este enfoque alinea tus nuevos datos con las propiedades estadísticas para las cuales los pesos del modelo fueron calibrados originalmente, asegurando un punto de partida estable y confiable para el entrenamiento.

Sin embargo, para obtener resultados óptimos, particularmente al entrenar un modelo desde cero, lo mejor es normalizar los datos usando sus propias estadísticas específicas. Cada dataset tiene una distribución única de colores y brillo, y calcular su media y desviación estándar precisas garantiza la normalización más exacta posible, lo que puede conducir a un mejor rendimiento del modelo.

Piensa en esto como recomendaciones sólidas, no como reglas estrictas. Mientras que usar las estadísticas de ImageNet es el enfoque más seguro para el fine-tuning, puedes usar las estadísticas de tu propio dataset. Hacerlo puede requerir un entrenamiento más extenso para que el modelo se adapte a la nueva distribución de datos. Por otro lado, para el entrenamiento desde cero, calcular tus propias estadísticas es esencial.

* La función `calculate_mean_std` itera a través de todo el dataset para acumular la suma y la suma de cuadrados de los valores de los píxeles para cada canal.
    * Estos valores acumulados se utilizan luego para calcular la media y la desviación estándar finales de todas las imágenes del dataset.

In [ ]:
def calculate_mean_std(dataset):
    """
    Calcula la media y la desviación estándar de un dataset de PyTorch.

    Args:
        dataset (torch.utils.data.Dataset): El dataset para el cual se 
                                            calcularán las estadísticas. Debe 
                                            devolver tensors de imagen.

    Returns:
        (torch.Tensor, torch.Tensor): Una tupla que contiene los tensors de media y
                                      desviación estándar, cada uno con 
                                      forma (C,).
    """
    # Crear un DataLoader para iterar a través del dataset en batches por eficiencia.
    # shuffle=False porque el orden de las imágenes no importa para este cálculo.
    loader = data.DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)

    # Inicializar tensors para almacenar la suma de los valores de los píxeles para cada canal (RGB).
    channel_sum = torch.zeros(3)
    # Inicializar tensors para almacenar la suma de los valores de los píxeles al cuadrado para cada canal.
    channel_sum_sq = torch.zeros(3)
    # Inicializar un contador para el número total de píxeles.
    num_pixels = 0

    # Envolver el loader con tqdm para crear una barra de progreso para el monitoreo.
    for images, _ in tqdm(loader, desc="Calculando Estadísticas del Dataset"):
        # Sumar el número total de píxeles en este batch al total acumulado.
        num_pixels += images.size(0) * images.size(2) * images.size(3)
        
        # Sumar los valores de los píxeles a lo largo de las dimensiones de batch, altura y ancho,
        # dejando solo la dimensión del canal. Sumar esto al total acumulado.
        channel_sum += images.sum(dim=[0, 2, 3])
        
        # Elevar al cuadrado cada valor de píxel, luego sumarlos de manera similar al paso anterior.
        channel_sum_sq += (images ** 2).sum(dim=[0, 2, 3])

    # Calcular la media para cada canal.
    mean = channel_sum / num_pixels
    # Calcular la desviación estándar usando la fórmula: sqrt(E[X^2] - E[X]^2)
    std = (channel_sum_sq / num_pixels - mean ** 2) ** 0.5

    # Devolver la media y la desviación estándar calculadas.
    return mean, std

<br>

Es hora de ver la función `calculate_mean_std` en acción. Ahora calcularás las estadísticas específicas para el **OxfordIIITPet dataset**, y utilizarás estos valores calculados en tu pipeline de augmentation final.

Pero antes de poder calcular las estadísticas, necesitas definir un pipeline de transformación simple.

**¿Por qué es necesario `simple_transform`?**

Tal vez te preguntes por qué necesitas transformar las imágenes solo para calcular su media. He aquí por qué esto es importante:


* **`transforms.Resize((100, 100))`**: Esto realiza dos funciones clave:
    * **Estandarización**: Asegura que cada imagen tenga exactamente las mismas dimensiones. Esto es importante para un cálculo justo, ya que evita que las imágenes más grandes o pequeñas sesguen el resultado general.
    * **Eficiencia**: Utilizas deliberadamente un tamaño *pequeño* (`(100, 100)`) como una optimización práctica. Procesar imágenes de `100x100` (10,000 píxeles) es mucho más rápido que usar un tamaño mayor como `224x224` (50,176 píxeles), y las estadísticas resultantes siguen siendo una excelente aproximación.
* **`transforms.ToTensor()`**: Esto es esencial porque tu función de cálculo opera con tensors numéricos, no con imágenes PIL. Esta transformación convierte las imágenes al formato requerido y escala sus valores de píxel al rango `[0.0, 1.0]`, que es el estándar para este tipo de computaciones.

In [ ]:
# Definir una transformación simple
simple_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor()
])

# Cargar el dataset OxfordIIITPet, aplicando la transformación simple a cada imagen
my_dataset = datasets.OxfordIIITPet(root=ox3_pet_data_path,
                                    split='test',                 # Especificar el uso del set de prueba (test set)
                                    download=ox3_pet_download,    # Descargar si no está presente
                                    transform=simple_transform    # Aplicar las transformaciones definidas
                                   )

# Calcular la media y la desviación estándar para el dataset
dataset_mean, dataset_std = calculate_mean_std(my_dataset)

print(f"\nCálculo Completado.")
print(f"Media del Dataset: {dataset_mean}")
print(f"Desviación Estándar del Dataset:  {dataset_std}")

<br>

Ahora que tienes la media y la desviación estándar específicas para el **OxfordIIITPet dataset**, abordemos una pregunta común sobre su reutilización.

**¿Son las estadísticas de imágenes de 100x100 lo suficientemente buenas para un pipeline de 224x224?**

**¡Sí, absolutamente!** Este es un enfoque pragmático y ampliamente utilizado.

Para la mayoría de las tareas de computer vision, las estadísticas calculadas a partir de una versión más pequeña y redimensionada de una imagen son un sustituto (proxy) muy sólido de la original. El sujeto principal (las mascotas) y la distribución del color siguen siendo los mismos. Si bien el cambio de tamaño introduce una cantidad mínima de cambio estadístico (debido a la interpolación de píxeles), esta diferencia es casi siempre insignificante en la práctica.

El beneficio es claro: obtienes una aceleración masiva en un cálculo que se realiza una sola vez, y las estadísticas resultantes son mucho más representativas de tu dataset que los valores genéricos (como los de ImageNet). Este es un caso clásico de una compensación (trade-off) que vale la pena entre la perfección teórica y la eficiencia práctica.

## Composing Transformations for Data Augmentation

El verdadero poder de las transformaciones proviene de encadenarlas. `transforms.Compose` crea un único pipeline que aplica una secuencia de transformaciones a una imagen en orden.


* Crearás dos pipelines:
    * `base_transform`: Un pipeline simple con solo los pasos esenciales de preprocesamiento (resizing y cropping). La normalización se omite intencionalmente para que las imágenes de salida sean visualmente correctas y puedan servir como una línea base limpia para la comparación.
    * `full_augmentation_pipeline`: Este incluye tus transformaciones aleatorias de aumento (augmentative transforms) y la normalización utilizando las estadísticas que acabas de calcular.

Esto te permitirá comparar directamente un batch "limpio" de imágenes con uno totalmente aumentado.

In [ ]:
# Una transformación simple para obtener una versión limpia y sin aumentos de las imágenes
base_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
    # Se omite la normalización para mantener los valores de píxel de la imagen en un rango fácil de visualizar.
])

# El pipeline de aumento completo con todas las transformaciones aleatorias
full_augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(size=224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.05, contrast=0.05),
    SaltAndPepperNoise(amount=0.001),
    transforms.ToTensor(),
    # Usando los valores de `mean` y `std` calculados en las imágenes de 100x100
    transforms.Normalize(mean=dataset_mean,
                         std=dataset_std)
])

### A Dataset Without Augmentations

Primero, echa un vistazo a los datos en su forma original.

* Usa un `DataLoader` con tu pipeline `base_transform` para cargar un batch de imágenes.
    * Esto aplica solo el resizing y cropping necesarios, dándote una línea base limpia y consistente para ver cómo lucen las imágenes antes de que se aplique cualquier augmentation aleatorio.

In [ ]:
# Load the dataset with ONLY the base transforms
original_dataset = datasets.OxfordIIITPet(root=ox3_pet_data_path, 
                                          split='test',
                                          download=ox3_pet_download,
                                          transform=base_transform
                                         )

# Create a DataLoader for the original images
original_loader = data.DataLoader(original_dataset, batch_size=9, shuffle=True)

In [ ]:
# Obtener un batch fijo de imágenes originales
original_images, _ = next(iter(original_loader))

# Crear una cuadrícula a partir del batch de imágenes, organizándolas con 3 imágenes por fila.
grid = vutils.make_grid(original_images, nrow=3, padding=2) 

print("Batch Original sin Aumentos:\n")
helper_utils.display_grid(grid)

### Applying the Augmentation Pipeline

Ahora verás el efecto del `full_augmentation_pipeline`. Tomarás el mismo batch de imágenes originales del paso anterior y aplicarás el pipeline completo a cada una. Harás esto en un bucle para ver cómo cambian los augmentations aleatorios con cada ejecución.

Esto demuestra el concepto central de data augmentation. Debido a que el pipeline incluye operaciones aleatorias, aplicarlo a una imagen produce un resultado ligeramente diferente cada vez. Esto es precisamente lo que sucede durante el entrenamiento del modelo cuando pasas un pipeline de augmentation a un `DataLoader`. Las transformaciones se aplican sobre la marcha ("on the fly"), por lo que con cada época (epoch), tu modelo recibe una versión única de la misma imagen original. Este proceso aumenta virtualmente el tamaño y la diversidad de tus datos de entrenamiento sin que tengas que recolectar más imágenes, lo que ayuda a tu modelo a generalizar mejor y reduce el overfitting.


**Nota:** los colores de la imagen pueden parecer diferentes porque cada canal de color ha sido normalizado en esta transformación. Siéntete libre de comentar el `Normalize` y observa la diferencia en las imágenes.

In [ ]:
# Usar un bucle para aplicar diferentes aumentos aleatorios
for i in range(3):
    
    augmented_batch = []
    
    # Iterar a través de cada imagen original en el batch fijo
    for img_tensor in original_images:
        
        # Convertir el tensor de nuevo a imagen PIL para aplicar transformaciones aleatorias
        img_pil = transforms.ToPILImage()(img_tensor)

        # Aplicar el pipeline de aumento aleatorio
        augmented_tensor = full_augmentation_pipeline(img_pil)

        # Añadir el tensor aumentado a la lista para su visualización
        augmented_batch.append(augmented_tensor)

    # Apilar la lista de tensors aumentados en un único tensor de batch
    final_batch = torch.stack(augmented_batch)

    # Crear una cuadrícula a partir del batch de imágenes, organizándolas con 3 imágenes por fila
    grid = vutils.make_grid(final_batch, nrow=3, padding=2)
    
    print(f"\nBatch Aumentado - Ejecución #{i + 1}")
    helper_utils.display_grid(grid)

## Conclusion

¡Felicidades! Has navegado con éxito por los componentes principales del preprocesamiento y la aumentación de imágenes con TorchVision.

Has visto de primera mano cómo construir un pipeline flexible y potente para preparar datos de imagen. Comenzaste con lo fundamental, como la conversión de imágenes entre el formato `Pillow (PIL)` y los `PyTorch Tensors`, y utilizaste utilidades esenciales como `make_grid` para una visualización efectiva.

A partir de ahí, exploraste las piezas fundamentales del data augmentation, aplicando transformaciones individuales como `RandomResizedCrop` y `ColorJitter`, e incluso definiendo una función de ruido personalizada desde cero. También viste la importancia de la **normalización** y la mejor práctica de calcular la media y la desviación estándar específicas para tu dataset, un paso clave para un entrenamiento estable y eficiente. Finalmente, reuniste todas estas técnicas usando `transforms.Compose` para crear un pipeline de aumento sofisticado que transforma las imágenes sobre la marcha.

Estas habilidades son fundamentales para prácticamente cualquier tarea de computer vision. Un pipeline de datos bien diseñado, complementado con aumentos cuidadosamente pensados, no solo hace que tu modelo sea más robusto, sino que también hace que todo tu flujo de trabajo sea más eficiente, reproducible y confiable.